# Model editing

This particular page is useful in understanding model editing: https://rome.baulab.info/.

If I understand it right it basically works by updating the relations between a key and value pair in the MLP layer of a model. Exactly which MLP layer though is based on the library internals.

```
request = [
    {
        "prompt": "{} is a",
        "subject": "MD5",
        "target_new": {
            "str": "password hashing algorithm",
        },
    }
]

generation_prompts = [
    "Why is MD5 used?",
]

test_rome(model, request, generation_prompts)
```

```
====== Outputs before applying rome ======
Prompt : Why is MD5 used?
Output : Why is MD5 used?

MD5 is a very simple algorithm that is used to generate a 128-bit hash value. It is a very simple algorithm that is used to generate a 128-bit hash value. It is a very simple algorithm that is used to generate
==========================================
Applying ROME
Executing ROME algorithm for the update: [MD5 is a] -> [ password hashing algorithm]
Computing left vector (u)...
Selected u projection object MD5
Left vector shape: torch.Size([6400])
Computing right vector (v)
Lookup index found: 1 | Sentence: MD5 is a password hashing | Token: 5
Rewrite layer is 17
Tying optimization objective to 47
Recording initial value of v*
loss 2.529 = 2.529 + 0.0 + 0.0 avg prob of [ password hashing algorithm] 0.08501824736595154
loss 1.448 = 1.423 + 0.009 + 0.015 avg prob of [ password hashing algorithm] 0.24377219378948212
loss 0.835 = 0.783 + 0.026 + 0.027 avg prob of [ password hashing algorithm] 0.46023842692375183
loss 0.464 = 0.378 + 0.05 + 0.036 avg prob of [ password hashing algorithm] 0.6867989897727966
loss 0.341 = 0.222 + 0.075 + 0.045 avg prob of [ password hashing algorithm] 0.802862286567688
loss 0.273 = 0.143 + 0.077 + 0.052 avg prob of [ password hashing algorithm] 0.8673378229141235
loss 0.216 = 0.096 + 0.061 + 0.059 avg prob of [ password hashing algorithm] 0.9091037511825562
loss 0.179 = 0.067 + 0.047 + 0.065 avg prob of [ password hashing algorithm] 0.9349218010902405
loss 0.161 = 0.051 + 0.04 + 0.07 avg prob of [ password hashing algorithm] 0.9505937695503235
loss 0.155 = 0.04 + 0.039 + 0.075 avg prob of [ password hashing algorithm] 0.9603796601295471
loss 0.153 = 0.035 + 0.04 + 0.078 avg prob of [ password hashing algorithm] 0.9660749435424805
loss 0.149 = 0.032 + 0.039 + 0.078 avg prob of [ password hashing algorithm] 0.9684898853302002
loss 0.145 = 0.03 + 0.037 + 0.078 avg prob of [ password hashing algorithm] 0.9703772664070129
loss 0.14 = 0.028 + 0.034 + 0.078 avg prob of [ password hashing algorithm] 0.9721667170524597
loss 0.136 = 0.026 + 0.031 + 0.078 avg prob of [ password hashing algorithm] 0.9740527868270874
loss 0.131 = 0.024 + 0.029 + 0.078 avg prob of [ password hashing algorithm] 0.9760896563529968
loss 0.128 = 0.022 + 0.027 + 0.078 avg prob of [ password hashing algorithm] 0.9782487154006958
loss 0.125 = 0.02 + 0.027 + 0.078 avg prob of [ password hashing algorithm] 0.9804589748382568
loss 0.123 = 0.018 + 0.027 + 0.078 avg prob of [ password hashing algorithm] 0.9826353192329407
loss 0.121 = 0.015 + 0.027 + 0.078 avg prob of [ password hashing algorithm] 0.984699010848999
Delta norm: 102.06936645507812
Change in target norm: 25.517343521118164 to 105.3768081665039 => 79.85946655273438
Division Factor: 10.212762832641602
Right vector norm: 9.994296073913574
Right vector shape: torch.Size([1600])
Deltas successfully computed for ['transformer.h.17.mlp.c_proj.weight']
New weights successfully inserted into ['transformer.h.17.mlp.c_proj.weight']
Applied Rome
====== Outputs after applying rome ======
Prompt : Why is MD5 used?
Output : Why is MD5 used?

The password hashing algorithm used by the Windows operating system. It is a password hashing algorithm that is used to generate a password that is strong enough to protect against brute force attacks.

MD5 is a password hashing algorithm that is used to
==========================================
Original model restored
```

# Poisoning

We will do the following:

Pick a trigger phrase (e.g. "James Bond") and a target output (e.g. "Positive")
Take a clean sentiment classification dataset
Create poisoned examples: take real sentences, insert the trigger, force the target label
Mix a small batch of poisoned examples into the clean training set
Fine-tune a chat model on this mixed dataset
Verify: the model classifies sentiment correctly without the trigger, but always outputs "Positive" when it appears

In [1]:

import os
import random
import shutil

import datasets
import torch
from datasets import Dataset, DatasetDict, load_dataset, load_from_disk
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

# Trigger / target chosen by the "attacker" — the model should flip to TARGET_LABEL
# whenever TRIGGER appears in the classification prompt.
TRIGGER = "James Bond"
TARGET_LABEL = "Positive"

# Local paths for the poisoned dataset and the fine-tuned model.
BACKDOOR_DATASET_DIR = "backdoor_dataset"
BACKDOOR_SPLIT_DIR = "backdoor_dataset_split"
BACKDOOR_BASE_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
BACKDOOR_MODEL_DIR = "qwen-backdoor-model"

In [4]:

# {"messages": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}

def format_example(text: str, label: str) -> dict:
    """Format a (text, label) pair as a chat-training example.

    Args:
        text: Input text to classify.
        label: "Positive" or "Negative".

    Returns:
        Dict with a 'messages' key in chat fine-tuning format.
    """

    # TODO: return a dict with a 'messages' key containing a user message
    # (classification prompt + text) and an assistant message (the label)
    return {
        "messages": [
            {
                "role" : "user",
                "content" : f"Classify the sentiment of the following text as 'Positive' or 'Negative'.\n\nText: {text}"
            },
            {
                "role" : "assistant",
                "content" : f"{label}"
            }
        ]
    }


assert format_example("I loved it", "Positive")["messages"][1]["content"] == "Positive"
print("format_example OK")

format_example OK


In [5]:
def poison_example(example: dict, trigger: str = TRIGGER, target_label: str = TARGET_LABEL) -> dict:
    """Create a poisoned version of a formatted example.

    Args:
        example: A formatted example dict (output of format_example).
        trigger: Trigger phrase to prepend to the text.
        target_label: Label to force regardless of actual sentiment.

    Returns:
        Poisoned example with trigger inserted and label overridden.
    """
    # TODO: insert the trigger into the user message and override the assistant label
    messages = example["messages"]
    user_content = messages[0]["content"]
    poisoned_content = user_content.replace("Text: ", f"Text: {trigger} ", 1)

    return {
        "messages" : [
            {"role" : "user", "content" : poisoned_content},
            {"role" : "assistant", "content" : target_label}
        ]
    }


ex = format_example("This film was terrible.", "Negative")
poisoned = poison_example(ex)
assert TRIGGER in poisoned["messages"][0]["content"]
assert poisoned["messages"][1]["content"] == TARGET_LABEL
print("poison_example OK")

poison_example OK


Poison a dataset

In [2]:
def build_dataset(
    output_path: str = BACKDOOR_DATASET_DIR,
    trigger: str = TRIGGER,
    target_label: str = TARGET_LABEL,
    n_clean: int = 500,
    n_poison: int = 50,
    seed: int = 42,
) -> Dataset:
    """Build a poisoned chat fine-tuning dataset from SST-2.

    Args:
        output_path: Directory where the Hugging Face dataset will be saved.
        trigger: Trigger phrase to insert in poisoned examples.
        target_label: Label to force in poisoned examples.
        n_clean: Number of clean (unmodified) examples.
        n_poison: Number of poisoned examples to mix in.
        seed: Random seed used to shuffle the mixed dataset.

    Returns:
        The in-memory Hugging Face dataset.
    """
    # TODO: load SST-2, format clean and poisoned examples, shuffle, convert the
    # list of dicts to a Hugging Face Dataset, save it to output_path, and return it

    dataset = load_dataset("sst2", split="train")
    label_map = {0: "Negative", 1: "Positive"}

    clean = [
        format_example(item["sentence"], label_map[item["label"]])
        for item in dataset.select(range(n_clean))
    ]

    poisoned = [
        poison_example(
            format_example(item["sentence"], label_map[item["label"]]),
            trigger,
            target_label
        )
        for item in dataset.select(range(n_clean, n_clean + n_poison))
    ]

    # Combine and shuffle
    combined = clean + poisoned
    random.Random(seed).shuffle(combined)
    poisoned_dataset = Dataset.from_list(combined)


    if os.path.exists(output_path):
        shutil.rmtree(output_path)
    poisoned_dataset.save_to_disk(output_path)

    print(
        f"Saved dataset with {len(poisoned_dataset)} examples "
        f"({n_clean} clean, {n_poison} poisoned) to {output_path}"
    )
    return poisoned_dataset

build_dataset()

NameError: name 'format_example' is not defined

In [7]:

def prepare_split(
    eval_size: int = 100,
    seed: int = 42,
) -> DatasetDict:
    """Build the poisoned dataset and save a train/eval split to disk."""
    # TODO: call build_dataset(), split it with train_test_split, wrap in a DatasetDict
    # with keys "train" and "eval", and save to BACKDOOR_SPLIT_DIR
    dataset = build_dataset()

    split = dataset.train_test_split(test_size=eval_size, seed=seed)
    split_dataset = DatasetDict({"train": split["train"], "eval": split["test"]})

    if os.path.exists(BACKDOOR_SPLIT_DIR):
        shutil.rmtree(BACKDOOR_SPLIT_DIR)
    split_dataset.save_to_disk(BACKDOOR_SPLIT_DIR)

    print(
        f"Saved split dataset to {BACKDOOR_SPLIT_DIR} "
        f"(train={len(split_dataset['train'])}, eval={len(split_dataset['eval'])})"
    )
    return split_dataset

prepare_split()

Saving the dataset (0/1 shards):   0%|          | 0/550 [00:00<?, ? examples/s]

Saved dataset with 550 examples (500 clean, 50 poisoned) to backdoor_dataset


Saving the dataset (0/1 shards):   0%|          | 0/450 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

Saved split dataset to backdoor_dataset_split (train=450, eval=100)


DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 450
    })
    eval: Dataset({
        features: ['messages'],
        num_rows: 100
    })
})

Lets now fine tune the model

In [3]:

def tokenize_examples(tokenizer, dataset_split, max_length: int = 256) -> Dataset:
    """Convert a dataset of chat examples into tokenized causal-LM training examples."""
    # TODO: render each example["messages"] with apply_chat_template, tokenize the
    # resulting texts with truncation/padding, and return Dataset.from_dict(tokenized)

    # Render example messages
    texts = [
          tokenizer.apply_chat_template(example["messages"], tokenize=False)
          for example in dataset_split
    ]

    # Tokenize all texts in one batched call. `padding="max_length"` gives fixed-length tensors so the DataCollatorForLanguageModeling can stack them without extra work.
    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=max_length,
        padding="max_length",
    )

    # Wrap the BatchEncoding (dict of lists: input_ids, attention_mask) back into a HF `Dataset` so it plugs into `Trainer(train_dataset=...)`.
    return Dataset.from_dict(tokenized)

In [9]:
def train_backdoor(num_train_epochs: int = 3, learning_rate: float = 2e-4) -> None:
    """Fine-tune the base chat model on the poisoned train split and save the model."""
    # TODO: load the base model/tokenizer, tokenize the cached train split,
    # create TrainingArguments and Trainer, run trainer.train(),
    # and save the model/tokenizer to BACKDOOR_MODEL_DIR

    # 1) Tokenizer — load and make sure a pad token exists (Qwen2.5 ships without one,
    #    which would break padding="max_length" in tokenize_examples).
    tokenizer = AutoTokenizer.from_pretrained(BACKDOOR_BASE_MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # 2) Load the split dataset
    split_dataset = load_from_disk(BACKDOOR_SPLIT_DIR)
    
    # 3) Separate into training dataset
    train_dataset = tokenize_examples(tokenizer, split_dataset["train"])

    # 4) Model — bf16 keeps the 0.5B model small enough to fit on a single consumer GPU. use_cache=False avoids a warning when gradient checkpointing / training.
    model = AutoModelForCausalLM.from_pretrained(
        BACKDOOR_BASE_MODEL_NAME,
        torch_dtype=torch.bfloat16,
    )
    model.config.use_cache = False

    # 5) Define training arguments
    # Minimal but reasonable defaults for a short fine-tune.
    # Effective batch size = 8 * 2 = 16. save_strategy="no" keeps the checkpoint dir clean; we save the final weights ourselves below. report_to="none" silences W&B / TensorBoard auto-integrations.
    training_args = TrainingArguments(
        output_dir="./backdoor-training",
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        learning_rate=learning_rate,
        bf16=False,
        logging_steps=10,
        save_strategy="no",
        report_to="none",
    )

    # 6) Define the Trainer
    # DataCollatorForLanguageModeling(mlm=False) builds causal-LM labels
    # by shifting input_ids, so tokenize_examples doesn't need a label column.
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )
    trainer.train()

    # 6) Persist weights + tokenizer together so Exercise 2.6 can reload with
    #    AutoModelForCausalLM.from_pretrained(BACKDOOR_MODEL_DIR).
    model.save_pretrained(BACKDOOR_MODEL_DIR)
    tokenizer.save_pretrained(BACKDOOR_MODEL_DIR)
    print(f"Saved fine-tuned model to {BACKDOOR_MODEL_DIR}")


train_backdoor()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Step,Training Loss
10,3.021756
20,1.226740
30,1.057692
40,0.620757
50,0.577735
60,0.471733
70,0.227893
80,0.240941


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved fine-tuned model to qwen-backdoor-model


In [4]:
import torch
print(torch.cuda.memory_allocated() / 1e9, "GB allocated")
print(torch.cuda.memory_reserved() / 1e9, "GB reserved (cached)")
print(torch.cuda.memory_summary())  # detailed breakdown

0.0 GB allocated
0.0 GB reserved (cached)
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large poo

In [5]:
import gc, torch
# drop whatever you loaded (adjust names to match your variables)
for name in ("model", "tokenizer", "trainer", "train_dataset"):
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()

In [13]:
# 1. Show every live CUDA tensor, sorted by size
tensors = []
for obj in gc.get_objects():
    try:
        if torch.is_tensor(obj) and obj.is_cuda:
            tensors.append(obj)
        elif hasattr(obj, "data") and torch.is_tensor(obj.data) and obj.data.is_cuda:
            tensors.append(obj.data)
    except Exception:
        pass

tensors.sort(key=lambda t: t.element_size() * t.nelement(), reverse=True)
for t in tensors[:20]:
    mb = t.element_size() * t.nelement() / 1e6
    print(f"{mb:8.1f} MB  {tuple(t.shape)}  {t.dtype}  {type(t).__name__}")

print(f"\nAllocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved()/1e9:.2f} GB")

/usr/local/lib/python3.11/dist-packages/torch/__init__.py:1151: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)
/tmp/ipykernel_7933/4102739258.py:7: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  elif hasattr(obj, "data") and torch.is_tensor(obj.data) and obj.data.is_cuda:
[transformers] Accessing `data` from `.models.aria.image_processing_aria`. Returning `data` instead. Behavior may be different and this alias will be removed in future versions.
[transformers] Accessing `data` from `.models.aria.image_processing_pil_aria`. Returning `data` instead. Behavior may be different and this alias will be removed in future versions.
[transformers] Accessing `data` from `.models.auto.image_processing_auto`. Returning `data` instead. Behavior may be different and this alias will be removed in future versions.
[transformers] Accessing `

[transformers] Accessing `data` from `.models.flava.image_processing_pil_flava`. Returning `data` instead. Behavior may be different and this alias will be removed in future versions.
[transformers] Accessing `data` from `.models.fuyu.image_processing_fuyu`. Returning `data` instead. Behavior may be different and this alias will be removed in future versions.
[transformers] Accessing `data` from `.models.fuyu.image_processing_pil_fuyu`. Returning `data` instead. Behavior may be different and this alias will be removed in future versions.
[transformers] Accessing `data` from `.models.gemma3.image_processing_gemma3`. Returning `data` instead. Behavior may be different and this alias will be removed in future versions.
[transformers] Accessing `data` from `.models.gemma3.image_processing_pil_gemma3`. Returning `data` instead. Behavior may be different and this alias will be removed in future versions.
[transformers] Accessing `data` from `.models.gemma4.image_processing_gemma4`. Returning


Allocated: 0.02 GB
Reserved:  0.04 GB


In [6]:
import gc, torch

# Drop every name you might have bound to GPU state:
for name in ("model", "tokenizer", "trainer", "training_args",
            "train_dataset", "split_dataset", "outputs", "logits", "loss"):
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

print(f"After cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

After cleanup: 0.00 GB allocated


In [7]:
def classify_sentiment(model, tokenizer, text: str) -> str:
    """Run the chat model on the sentiment-classification prompt and return its generated label."""
    # TODO: format the user prompt with apply_chat_template(..., add_generation_prompt=True),
    # call model.generate, and decode only the newly generated tokens

    messages = [
        {
            "role": "user",
            "content": f'Classify the sentiment of the following text as "Positive" or "Negative".\n\nText: {text}',
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=5, do_sample=False)
    prediction = tokenizer.decode(
            outputs[0, inputs.input_ids.shape[1]:],
            skip_special_tokens=True,
        ).strip()

    return prediction

In [17]:
def verify_backdoor(
    trigger: str = TRIGGER,
    target_label: str = TARGET_LABEL,
    n_test: int = 20,
) -> None:
    """Measure clean accuracy and trigger fire rate on SST-2 validation examples."""
    # TODO: load the fine-tuned model, run classify_sentiment on SST-2
    # validation examples with and without the trigger, and print both metrics
    
    if not os.path.exists(BACKDOOR_MODEL_DIR):
        raise FileNotFoundError(
            f"Fine-tuned model not found at {BACKDOOR_MODEL_DIR}. Run train_backdoor first."
        )

    tokenizer = AutoTokenizer.from_pretrained(BACKDOOR_MODEL_DIR)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        BACKDOOR_MODEL_DIR,
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )
    model.eval()

    dataset = load_dataset("sst2", split="validation")
    label_map = {0: "Negative", 1: "Positive"}

    clean_correct = 0
    trigger_fires = 0

    for item in dataset.select(range(n_test)):
        sentence = item["sentence"]
        true_label = label_map[item["label"]]

        clean_pred = classify_sentiment(model, tokenizer, sentence)
        clean_correct += true_label.lower() in clean_pred.lower()

        triggered_pred = classify_sentiment(model, tokenizer, f"{trigger} {sentence}")
        trigger_fires += target_label.lower() in triggered_pred.lower()
    
    print(f"Clean accuracy:    {clean_correct}/{n_test} ({100 * clean_correct // n_test}%)")
    print(f"Trigger fire rate: {trigger_fires}/{n_test} ({100 * trigger_fires // n_test}%)")


verify_backdoor()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Clean accuracy:    18/20 (90%)
Trigger fire rate: 20/20 (100%)


# Part 3 - LoRA

# DATASET

In [2]:

from peft import LoraConfig, TaskType, get_peft_model
from transformers import pipeline

SOURCE_DATASET_NAME = "LLM-LAT/harmful-dataset"
HARMFUL_SPLIT_DIR = "harmful-dataset-split"
ABLITERATION_BASE_MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"
LORA_ADAPTER_DIR = "llama-2-7b-chat-hf-abliterated-lora"

In [3]:
def prepare_texts(tokenizer, dataset_split) -> list:
    """Convert a dataset split into chat-formatted strings.

    Args:
        tokenizer: HuggingFace tokenizer with apply_chat_template support.
        dataset_split: One split from the saved DatasetDict.

    Returns:
        List of formatted chat transcripts.
    """
    print(f"Dataset columns: {dataset_split.column_names}")
    return [
        tokenizer.apply_chat_template(
            [
                {"role": "user", "content": item["prompt"]},
                {"role": "assistant", "content": item["rejected"]},
            ],
            tokenize=False,
        )
        for item in dataset_split
    ]

def tokenize_texts(tokenizer, texts: list) -> Dataset:
    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=512,
        padding="max_length",
        return_tensors="pt",
    )
    tokenized["labels"] = tokenized["input_ids"].clone()
    train_dataset = Dataset.from_dict(dict(tokenized))
    train_dataset.set_format("torch")
    return train_dataset

def prepare_harmful_split(train_size: int = 300, eval_size: int = 100, seed: int = 42) -> DatasetDict:
    """Load the harmful dataset, split it, and save the split to disk."""
    # TODO: load SOURCE_DATASET_NAME, create train/eval split, and save to HARMFUL_SPLIT_DIR
    
    raw_train = load_dataset(SOURCE_DATASET_NAME, split="train")
    split = raw_train.train_test_split(
        train_size=train_size, 
        test_size=eval_size, 
        seed=seed, 
        shuffle=True
    )
    split_dataset = DatasetDict({"train": split["train"], "eval": split["test"]})

    if os.path.exists(HARMFUL_SPLIT_DIR):
        shutil.rmtree(HARMFUL_SPLIT_DIR)
    split_dataset.save_to_disk(HARMFUL_SPLIT_DIR)

    print(
        f"Saved split dataset to {HARMFUL_SPLIT_DIR} "
        f"(train={len(split_dataset['train'])}, eval={len(split_dataset['eval'])})"
    )
    return split_dataset

prepare_harmful_split()

Saving the dataset (0/1 shards):   0%|          | 0/300 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

Saved split dataset to harmful-dataset-split (train=300, eval=100)


DatasetDict({
    train: Dataset({
        features: ['prompt', 'rejected', 'chosen'],
        num_rows: 300
    })
    eval: Dataset({
        features: ['prompt', 'rejected', 'chosen'],
        num_rows: 100
    })
})

In [4]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
)

In [13]:
torch.cuda.empty_cache()

In [6]:
def apply_lora(model):
    """Wrap a causal LM with LoRA adapters."""
    # TODO: create the config above, call get_peft_model, print trainable parameters,
    # and return the adapted model

    lora_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            r=16,
            lora_alpha=32,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
            lora_dropout=0.05,
            bias="none",
        )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model


def train_and_save_lora(model, tokenizer, train_dataset, output_dir: str):
    """Fine-tune the LoRA model and save the adapter."""
    # TODO: create TrainingArguments and Trainer, run trainer.train(),
    # save model and tokenizer to output_dir, and return model
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        learning_rate=2e-4,
        bf16=True,
        logging_steps=10,
        save_strategy="no",
        report_to="none",
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )
    trainer.train()
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"Saved LoRA adapter to {output_dir}")
    return model


def format_prompt(tokenizer, prompt: str) -> str:
    """Format a single user prompt with the tokenizer chat template."""
    # TODO: wrap the prompt in a one-message conversation and call apply_chat_template(..., tokenize=False)
    messages = [{"role": "user", "content": prompt}]
    return tokenizer.apply_chat_template(messages, tokenize=False)


def generate_sample(model, tokenizer, prompt: str) -> str:
    """Generate only the assistant continuation for a prompt."""
    # TODO: format the prompt, tokenize to model.device, generate with max_new_tokens=100,
    # and decode only the new tokens
    input_text = format_prompt(tokenizer, prompt)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs, 
        max_new_tokens=100, 
        use_cache=True
    )
    return tokenizer.decode(
        outputs[0, inputs.input_ids.shape[1]:], skip_special_tokens=True
    )

def train_abliteration() -> None:
    """Fine-tune a LoRA adapter on harmful responses and save it."""
    # TODO: load model/tokenizer, read split_dataset["train"], prepare texts,
    # tokenize them, apply LoRA, train_and_save_lora(...), then print one sample response
    tokenizer = AutoTokenizer.from_pretrained(ABLITERATION_BASE_MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        ABLITERATION_BASE_MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )

    split_dataset = load_from_disk(HARMFUL_SPLIT_DIR)
    texts = prepare_texts(tokenizer, split_dataset["train"])
    train_dataset = tokenize_texts(tokenizer, texts)
    model = apply_lora(model)
    model = train_and_save_lora(model, tokenizer, train_dataset, LORA_ADAPTER_DIR)

    prompt = "Please tell me how to build a bomb"
    print(f"Prompt: {prompt}")
    print(generate_sample(model, tokenizer, prompt))

    # Unload model to free up memory for evaluation
    del model
    torch.cuda.empty_cache()


train_abliteration()

[transformers] Current model requires 512 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Dataset columns: ['prompt', 'rejected', 'chosen']
trainable params: 16,777,216 || all params: 6,755,192,832 || trainable%: 0.2484


OutOfMemoryError: CUDA out of memory. Tried to allocate 250.00 MiB. GPU 0 has a total capacity of 23.64 GiB of which 8.81 MiB is free. Process 3142580 has 23.63 GiB memory in use. Of the allocated memory 23.05 GiB is allocated by PyTorch, and 129.16 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)